# 03 — Cross-Architecture Comparison

Consumes the saved CSVs from `01_sg_population.ipynb` and `02_st_population.ipynb` and produces:

- Per `(regime, mt)` SG-vs-ST scatter (`plot_sg_vs_st_scatter`) and pair-matrix diff (`plot_pair_matrix_diff`) figures
- SPEC §11 acceptance check matrix:
  - **C3** — mt=2 vs mt=4 stability per regime/arch (ρ ≥ 0.5)
  - **C4** — SG 3-way estimator agreement (ρ ≥ 0.4 between any two of {gnn, captum_ig, attention})
  - **C5** — ST native vs supplementary (ρ ≥ 0.4)
  - **C6** — biological-prior overlap with `{S1_D1, S5_D5, S3_D3, S2_D1, S4_D5, S4_D7}` (≥ 2 in top-10, *advisory*)
- `research/xai/cross_arch_comparison.{csv, md, npy}` — aggregated SG-vs-ST summary

> **C2 (across-fold stability) is not computed here** because `run_sg` / `run_st` aggregate folds into a single PopulationResult per cell. Computing C2 needs per-fold rankings, which would require splitting the population sweep into per-fold output dirs — a future enhancement.

Reference: [`docs/SPEC_xai_graph.md`](../../../docs/SPEC_xai_graph.md) (rev. 4).

In [1]:
%matplotlib inline
import os, sys, json
from pathlib import Path
from typing import Dict, Optional

import numpy as np
import pandas as pd
import torch
from scipy import stats

PROJECT_ROOT = Path(os.getcwd()).resolve().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.xai import (
    PopulationResult,
    plot_sg_vs_st_scatter, plot_pair_matrix_diff,
    CHANNEL_NAMES, N_CH,
)

XAI_ROOT = PROJECT_ROOT / "research/xai"
CROSS_OUT = XAI_ROOT / "cross_arch"
CROSS_OUT.mkdir(parents=True, exist_ok=True)

# Defaults the cross-arch comparison runs against. Edit these to compare a
# different SG estimator, ST path, or chromophore.
SG_PRIMARY_ESTIMATOR = "gnn"           # 'gnn' | 'captum_ig' | 'attention'
ST_PRIMARY_PATH      = "native"        # 'native' | 'supplementary'
ST_PRIMARY_HB        = "hbr"           # 'hbo' | 'hbr' | 'hbt'  (rev. 6 — paper headline)
ST_HBS               = ("hbo", "hbr", "hbt")  # all 3 chromophores swept by 02_st_population

# Biological-prior channels per SPEC §11 C6 (advisory).
BIO_PRIOR = {"S1_D1", "S5_D5", "S3_D3", "S2_D1", "S4_D5", "S4_D7"}

print(f"XAI_ROOT               = {XAI_ROOT}")
print(f"SG primary estimator   = {SG_PRIMARY_ESTIMATOR}")
print(f"ST primary path        = {ST_PRIMARY_PATH}")
print(f"ST primary chromophore = {ST_PRIMARY_HB} (sweeping {ST_HBS})")
print(f"Biological prior       = {sorted(BIO_PRIOR)}")

XAI_ROOT               = /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/research/xai
SG primary estimator   = gnn
ST primary path        = native
ST primary chromophore = hbr (sweeping ('hbo', 'hbr', 'hbt'))
Biological prior       = ['S1_D1', 'S2_D1', 'S3_D3', 'S4_D5', 'S4_D7', 'S5_D5']


## Helpers

In [2]:
REGIMES = ("kfold-5", "kfold-10", "loso")
MTS = (2, 4)
SG_ESTS = ("gnn", "captum_ig", "attention")
ST_PATHS = ("native", "supplementary")


def _try_load(in_dir: Path) -> Optional[PopulationResult]:
    """Return PopulationResult.from_csv(in_dir) or None if not present."""
    if not (in_dir / "node_importance.csv").is_file():
        return None
    try:
        return PopulationResult.from_csv(in_dir)
    except Exception as e:
        print(f"  [warn] failed to load {in_dir}: {e}")
        return None


def load_sg_results() -> Dict[tuple, PopulationResult]:
    """Returns {(regime, mt, estimator): PopulationResult} for everything found."""
    out = {}
    for regime in REGIMES:
        for mt in MTS:
            for est in SG_ESTS:
                d = XAI_ROOT / f"sg/{regime}/mt{mt}/{est}"
                r = _try_load(d)
                if r is not None:
                    out[(regime, mt, est)] = r
    return out


def load_st_results() -> Dict[tuple, PopulationResult]:
    """Returns {(regime, mt, hb, path): PopulationResult} for everything found.

    rev. 6 — keys gained `hb` element after ST output paths gained {hb}/ layer
    (research/xai/st/{hb}/{regime}/mt{N}/{path}/...).
    """
    out = {}
    for hb in ST_HBS:
        for regime in REGIMES:
            for mt in MTS:
                for path in ST_PATHS:
                    d = XAI_ROOT / f"st/{hb}/{regime}/mt{mt}/{path}"
                    r = _try_load(d)
                    if r is not None:
                        out[(regime, mt, hb, path)] = r
    return out


def spearman_top_k_rho(a: np.ndarray, b: np.ndarray) -> float:
    ra = (-a).argsort().argsort()
    rb = (-b).argsort().argsort()
    rho, _ = stats.spearmanr(ra, rb)
    return float(rho)


def jaccard_top_k(a: np.ndarray, b: np.ndarray, k: int) -> float:
    ai = set(np.argsort(-a)[:k].tolist())
    bi = set(np.argsort(-b)[:k].tolist())
    union = ai | bi
    return len(ai & bi) / len(union) if union else 0.0


def c6_biological_prior(channel_importance: np.ndarray, k: int = 10) -> tuple:
    top_idx = np.argsort(-channel_importance)[:k]
    top_names = {CHANNEL_NAMES[i] for i in top_idx}
    overlap = top_names & BIO_PRIOR
    return overlap, len(overlap) >= 2

## Load all available results

Discovers everything currently saved under `research/xai/sg/` and `research/xai/st/`. Cells you haven't run yet are skipped silently.

In [3]:
sg_all = load_sg_results()
st_all = load_st_results()

print(f"SG results loaded: {len(sg_all)} / {len(REGIMES) * len(MTS) * len(SG_ESTS)} possible")
for k in sg_all:
    print(f"  {k}")
print()
print(f"ST results loaded: {len(st_all)} / {len(REGIMES) * len(MTS) * len(ST_HBS) * len(ST_PATHS)} possible")
for k in st_all:
    print(f"  {k}")

SG results loaded: 18 / 18 possible
  ('kfold-5', 2, 'gnn')
  ('kfold-5', 2, 'captum_ig')
  ('kfold-5', 2, 'attention')
  ('kfold-5', 4, 'gnn')
  ('kfold-5', 4, 'captum_ig')
  ('kfold-5', 4, 'attention')
  ('kfold-10', 2, 'gnn')
  ('kfold-10', 2, 'captum_ig')
  ('kfold-10', 2, 'attention')
  ('kfold-10', 4, 'gnn')
  ('kfold-10', 4, 'captum_ig')
  ('kfold-10', 4, 'attention')
  ('loso', 2, 'gnn')
  ('loso', 2, 'captum_ig')
  ('loso', 2, 'attention')
  ('loso', 4, 'gnn')
  ('loso', 4, 'captum_ig')
  ('loso', 4, 'attention')

ST results loaded: 36 / 36 possible
  ('kfold-5', 2, 'hbo', 'native')
  ('kfold-5', 2, 'hbo', 'supplementary')
  ('kfold-5', 4, 'hbo', 'native')
  ('kfold-5', 4, 'hbo', 'supplementary')
  ('kfold-10', 2, 'hbo', 'native')
  ('kfold-10', 2, 'hbo', 'supplementary')
  ('kfold-10', 4, 'hbo', 'native')
  ('kfold-10', 4, 'hbo', 'supplementary')
  ('loso', 2, 'hbo', 'native')
  ('loso', 2, 'hbo', 'supplementary')
  ('loso', 4, 'hbo', 'native')
  ('loso', 4, 'hbo', 'supplemen

## SPEC §11 acceptance summary

In [4]:
print("=" * 80)
print("SPEC §11 C3 — mt=2 vs mt=4 stability (ρ ≥ 0.5)")
print("=" * 80)
for regime in REGIMES:
    for est in SG_ESTS:
        r2 = sg_all.get((regime, 2, est))
        r4 = sg_all.get((regime, 4, est))
        if r2 is None or r4 is None:
            continue
        rho = spearman_top_k_rho(r2.channel_importance_mean, r4.channel_importance_mean)
        flag = "✓" if rho >= 0.5 else "✗"
        print(f"  SG {regime:>9s} {est:>10s}: ρ(mt2, mt4) = {rho:+.3f}   {flag}")
    for hb in ST_HBS:
        for path in ST_PATHS:
            r2 = st_all.get((regime, 2, hb, path))
            r4 = st_all.get((regime, 4, hb, path))
            if r2 is None or r4 is None:
                continue
            rho = spearman_top_k_rho(r2.channel_importance_mean, r4.channel_importance_mean)
            flag = "✓" if rho >= 0.5 else "✗"
            print(f"  ST {regime:>9s} {hb:>3s} {path:>13s}: ρ(mt2, mt4) = {rho:+.3f}   {flag}")

print()
print("=" * 80)
print("SPEC §11 C4 — SG 3-way estimator agreement (ρ ≥ 0.4 between any two)")
print("=" * 80)
for regime in REGIMES:
    for mt in MTS:
        ests = [(est, sg_all.get((regime, mt, est))) for est in SG_ESTS]
        ests = [(e, r) for e, r in ests if r is not None]
        for i, (a, ra) in enumerate(ests):
            for b, rb in ests[i + 1:]:
                rho = spearman_top_k_rho(ra.channel_importance_mean, rb.channel_importance_mean)
                flag = "✓" if rho >= 0.4 else "✗"
                print(f"  {regime:>9s} mt={mt}  ρ({a:>10s}, {b:>10s}) = {rho:+.3f}   {flag}")

print()
print("=" * 80)
print("SPEC §11 C5 — ST native vs supplementary (ρ ≥ 0.4)")
print("=" * 80)
for regime in REGIMES:
    for mt in MTS:
        for hb in ST_HBS:
            n = st_all.get((regime, mt, hb, "native"))
            s = st_all.get((regime, mt, hb, "supplementary"))
            if n is None or s is None:
                continue
            rho = spearman_top_k_rho(n.channel_importance_mean, s.channel_importance_mean)
            flag = "✓" if rho >= 0.4 else "✗"
            print(f"  {regime:>9s} mt={mt} {hb:>3s}: ρ(native, supplementary) = {rho:+.3f}   {flag}")

print()
print("=" * 80)
print("SPEC §11 C6 — biological-prior overlap in top-10 (≥ 2 of " + str(sorted(BIO_PRIOR)) + ", *advisory*)")
print("=" * 80)
for (regime, mt, est), r in sg_all.items():
    overlap, passes = c6_biological_prior(r.channel_importance_mean)
    flag = "✓" if passes else "✗"
    print(f"  SG {regime:>9s} mt={mt} {est:>10s}: top-10 ∩ prior = {sorted(overlap)}   {flag}")
for (regime, mt, hb, path), r in st_all.items():
    overlap, passes = c6_biological_prior(r.channel_importance_mean)
    flag = "✓" if passes else "✗"
    print(f"  ST {regime:>9s} mt={mt} {hb:>3s} {path:>13s}: top-10 ∩ prior = {sorted(overlap)}   {flag}")

SPEC §11 C3 — mt=2 vs mt=4 stability (ρ ≥ 0.5)
  SG   kfold-5        gnn: ρ(mt2, mt4) = +0.379   ✗
  SG   kfold-5  captum_ig: ρ(mt2, mt4) = +0.355   ✗
  SG   kfold-5  attention: ρ(mt2, mt4) = +0.698   ✓
  ST   kfold-5 hbo        native: ρ(mt2, mt4) = +0.510   ✓
  ST   kfold-5 hbo supplementary: ρ(mt2, mt4) = +0.849   ✓
  ST   kfold-5 hbr        native: ρ(mt2, mt4) = +0.589   ✓
  ST   kfold-5 hbr supplementary: ρ(mt2, mt4) = +0.823   ✓
  ST   kfold-5 hbt        native: ρ(mt2, mt4) = +0.600   ✓
  ST   kfold-5 hbt supplementary: ρ(mt2, mt4) = +0.855   ✓
  SG  kfold-10        gnn: ρ(mt2, mt4) = +0.527   ✓
  SG  kfold-10  captum_ig: ρ(mt2, mt4) = +0.339   ✗
  SG  kfold-10  attention: ρ(mt2, mt4) = +0.708   ✓
  ST  kfold-10 hbo        native: ρ(mt2, mt4) = +0.564   ✓
  ST  kfold-10 hbo supplementary: ρ(mt2, mt4) = +0.805   ✓
  ST  kfold-10 hbr        native: ρ(mt2, mt4) = +0.520   ✓
  ST  kfold-10 hbr supplementary: ρ(mt2, mt4) = +0.854   ✓
  ST  kfold-10 hbt        native: ρ(mt2, mt4) = +0.

## SG-vs-ST cross-architecture comparison per `(regime, mt)` cell

Uses the configured primary estimator (`SG_PRIMARY_ESTIMATOR`) and primary path (`ST_PRIMARY_PATH`). Saves figures to `research/xai/cross_arch/{regime}_mt{N}/`.

In [5]:
rows = []
diffs_per_cell = {}
for regime in REGIMES:
    for mt in MTS:
        sg = sg_all.get((regime, mt, SG_PRIMARY_ESTIMATOR))
        if sg is None:
            continue
        for hb in ST_HBS:
            st = st_all.get((regime, mt, hb, ST_PRIMARY_PATH))
            if st is None:
                continue
            cell_dir = CROSS_OUT / f"{regime}_mt{mt}_{hb}"
            cell_dir.mkdir(parents=True, exist_ok=True)

            plot_sg_vs_st_scatter(sg, st, cell_dir)
            plot_pair_matrix_diff(sg, st, cell_dir)

            rho_ch = spearman_top_k_rho(sg.channel_importance_mean, st.channel_importance_mean)
            j5 = jaccard_top_k(sg.channel_importance_mean, st.channel_importance_mean, k=5)
            j10 = jaccard_top_k(sg.channel_importance_mean, st.channel_importance_mean, k=10)
            rho_pair = float(stats.spearmanr(
                sg.pair_matrix[~np.eye(N_CH, dtype=bool)],
                st.pair_matrix[~np.eye(N_CH, dtype=bool)],
            ).statistic)

            rows.append({
                "regime": regime, "mt": mt, "hb": hb,
                "sg_estimator": SG_PRIMARY_ESTIMATOR,
                "st_path": ST_PRIMARY_PATH,
                "rho_channel_rank": rho_ch,
                "jaccard_top5": j5,
                "jaccard_top10": j10,
                "rho_pair_offdiag": rho_pair,
                "n_sg_subjects": sg.n_subjects, "n_sg_trials": sg.n_trials,
                "n_st_subjects": st.n_subjects, "n_st_trials": st.n_trials,
            })

            mask = ~np.eye(N_CH, dtype=bool)
            sg_z = (sg.pair_matrix - sg.pair_matrix[mask].mean()) / (sg.pair_matrix[mask].std(ddof=0) + 1e-12)
            st_z = (st.pair_matrix - st.pair_matrix[mask].mean()) / (st.pair_matrix[mask].std(ddof=0) + 1e-12)
            diffs_per_cell[(regime, mt, hb)] = (sg_z - st_z).astype(np.float32)

            print(f"  {regime:>9s} mt={mt} {hb:>3s}: ρ_channel={rho_ch:+.3f}  J@5={j5:.2f}  J@10={j10:.2f}  ρ_pair={rho_pair:+.3f}")

cmp_df = pd.DataFrame(rows)
print()
print(cmp_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    kfold-5 mt=2 hbo: ρ_channel=+0.168  J@5=0.11  J@10=0.33  ρ_pair=+0.391
    kfold-5 mt=2 hbr: ρ_channel=-0.089  J@5=0.11  J@10=0.25  ρ_pair=+0.261
    kfold-5 mt=2 hbt: ρ_channel=-0.020  J@5=0.11  J@10=0.33  ρ_pair=+0.374
    kfold-5 mt=4 hbo: ρ_channel=-0.176  J@5=0.11  J@10=0.33  ρ_pair=+0.251
    kfold-5 mt=4 hbr: ρ_channel=-0.095  J@5=0.11  J@10=0.33  ρ_pair=+0.212
    kfold-5 mt=4 hbt: ρ_channel=-0.019  J@5=0.11  J@10=0.33  ρ_pair=+0.303
   kfold-10 mt=2 hbo: ρ_channel=+0.090  J@5=0.25  J@10=0.25  ρ_pair=+0.352
   kfold-10 mt=2 hbr: ρ_channel=+0.134  J@5=0.25  J@10=0.25  ρ_pair=+0.309
   kfold-10 mt=2 hbt: ρ_channel=+0.008  J@5=0.11  J@10=0.18  ρ_pair=+0.325
   kfold-10 mt=4 hbo: ρ_channel=-0.285  J@5=0.25  J@10=0.33  ρ_pair=+0.255
   kfold-10 mt=4 hbr: ρ_channel=-0.345  J@5=0.00  J@10=0.25  ρ_pair=+0.212
   kfold-10 mt=4 hbt: ρ_channel=-0.078  J@5=0.25  J@10=0.33  ρ_pair=+0.230
       loso mt=2 hbo: ρ_channel=+0.060  J@5=0.11  J@10=0.43  ρ_pair=+0.443
       loso mt=2 hbr: ρ_c

## Persist cross-arch summary

Writes:
- `research/xai/cross_arch_comparison.csv` — one row per (regime, mt)
- `research/xai/cross_arch_comparison.md` — human-readable digest
- `research/xai/cross_arch_pair_diffs.npy` — stacked (n_cells, 23, 23) z-scored diff matrices

In [6]:
COMP_CSV = XAI_ROOT / "cross_arch_comparison.csv"
COMP_MD  = XAI_ROOT / "cross_arch_comparison.md"
COMP_NPY = XAI_ROOT / "cross_arch_pair_diffs.npy"

cmp_df.to_csv(COMP_CSV, index=False, float_format="%.6f")
print(f"Wrote {COMP_CSV.relative_to(PROJECT_ROOT)}  ({len(cmp_df)} rows)")

if diffs_per_cell:
    keys = sorted(diffs_per_cell.keys())
    stack = np.stack([diffs_per_cell[k] for k in keys], axis=0)
    np.save(COMP_NPY, stack)
    keys_path = COMP_NPY.with_suffix(".keys.json")
    keys_path.write_text(json.dumps([list(k) for k in keys]))
    print(f"Wrote {COMP_NPY.relative_to(PROJECT_ROOT)}  shape={stack.shape}")
    print(f"Wrote {keys_path.relative_to(PROJECT_ROOT)}")

md_lines = ["# Cross-architecture XAI comparison\n",
            f"SG estimator: `{SG_PRIMARY_ESTIMATOR}`  ·  ST path: `{ST_PRIMARY_PATH}`\n"]
md_lines.append("\n## Per-cell summary\n")
md_lines.append(cmp_df.to_markdown(index=False, floatfmt=".3f"))
COMP_MD.write_text("\n".join(md_lines))
print(f"Wrote {COMP_MD.relative_to(PROJECT_ROOT)}")

Wrote research/xai/cross_arch_comparison.csv  (18 rows)
Wrote research/xai/cross_arch_pair_diffs.npy  shape=(18, 23, 23)
Wrote research/xai/cross_arch_pair_diffs.keys.json
Wrote research/xai/cross_arch_comparison.md


## Done

`research/xai/cross_arch_comparison.{csv, md, npy}` and per-cell `cross_arch/{regime}_mt{N}/fig_*.{png,svg}` are now ready for paper inclusion. The acceptance summary printed above is the SPEC §11 evaluation; copy the relevant rows into the paper's discussion section.

To compare a different SG estimator or ST path, change `SG_PRIMARY_ESTIMATOR` / `ST_PRIMARY_PATH` near the top of this notebook and rerun from cell 6 (load) onwards.